# 01 · Load, Validate & Clean — Doomscroll

**Goal:** turn the raw KuaiSAR files into one clean, canonical event table that notebooks 02 and 03 build on.

| | |
|---|---|
| **Reads** | `data/raw/**` (feed + search interactions, user features, item features) |
| **Writes** (`data/interim/`) | `events.parquet`, `items.parquet`, `users.parquet`, `schema.json`, `cleaning_report.csv` |
| **Figures** | `eda/figures/01_*.png` |

**Plan**
1. Locate files and inspect schemas
2. Map raw column names to canonical names (`user, item, ts, click, like, ...`) — *review this table once*
3. Load with memory-friendly dtypes
4. Profile and audit data quality
5. Clean (every step is logged with rows removed)
6. Save to parquet

> Raw data stays in `data/raw` and is git-ignored. Nothing in this notebook writes outside `data/` and `eda/figures/`.

In [ ]:
import json, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 170)
sns.set_theme(style="whitegrid", context="notebook")

ROOT = Path.cwd().parent if Path.cwd().name == "eda" else Path.cwd()
RAW, INTERIM, FIG = ROOT / "data" / "raw", ROOT / "data" / "interim", ROOT / "eda" / "figures"
for p in (INTERIM, FIG):
    p.mkdir(parents=True, exist_ok=True)

def savefig(name):
    plt.tight_layout(); plt.savefig(FIG / f"{name}.png", dpi=150, bbox_inches="tight"); plt.show()

print("ROOT:", ROOT)

## 1 · Locate files

In [ ]:
EXTS = {".csv", ".tsv", ".gz", ".parquet"}
all_files = sorted(p for p in RAW.rglob("*") if p.is_file() and p.suffix in EXTS)
assert all_files, f"No data files under {RAW}. Download KuaiSAR first (see data/README.md)."
display(pd.DataFrame({"file": [str(p.relative_to(RAW)) for p in all_files],
                      "MB": [round(p.stat().st_size / 1e6, 1) for p in all_files]}))

# regex on the lower-cased file name -> role. Adjust if the real file names differ.
ROLE_PATTERNS = {
    "rec":  r"rec.*inter|inter.*rec",
    "src":  r"src.*inter|search.*inter|inter.*(src|search)",
    "user": r"user.*feat",
    "item": r"item.*feat",
}
FILES_OVERRIDE = {}   # e.g. {"rec": [RAW / "KuaiSAR" / "rec_inter.csv"]}

FILES = {role: FILES_OVERRIDE.get(role, [p for p in all_files if re.search(pat, p.stem.lower())])
         for role, pat in ROLE_PATTERNS.items()}
for role, fs in FILES.items():
    print(f"{role:5s} ->", [f.name for f in fs] or "NOT FOUND")
assert FILES["rec"] or FILES["src"], "No interaction file matched. Set FILES_OVERRIDE above."

## 2 · Inspect raw schemas

In [ ]:
def read_any(path, nrows=None, usecols=None):
    path = Path(path)
    if path.suffix == ".parquet":
        df = pd.read_parquet(path, columns=usecols)
        return df.head(nrows) if nrows else df
    sep = "\t" if ".tsv" in path.name else ","
    return pd.read_csv(path, sep=sep, nrows=nrows, usecols=usecols, low_memory=False)

HEADS = {}
for role, fs in FILES.items():
    if fs:
        HEADS[role] = read_any(fs[0], nrows=5)
        print(f"\n=== {role}: {fs[0].name}  ({HEADS[role].shape[1]} columns)")
        display(HEADS[role])

## 3 · Map raw columns → canonical names
The resolver below tries common names. **Read the printed table once.** If anything important is `—` or wrong, put the right raw name in `OVERRIDES` and re-run.

In [ ]:
CANDIDATES = {
    "user": ["user_id", "uid", "userid", "user"],
    "item": ["item_id", "pos_item_id", "video_id", "photo_id", "iid", "vid", "item"],
    "ts": ["timestamp", "time", "ts", "time_ms", "datetime", "date_time", "log_time"],
    "query": ["query_id", "qid", "query"],
    "click": ["click", "is_click", "clicked", "click_cnt"],
    "like": ["like", "is_like", "liked", "like_cnt"],
    "follow": ["follow", "is_follow", "followed", "follow_cnt"],
    "comment": ["comment", "is_comment", "commented", "comment_cnt"],
    "forward": ["forward", "share", "is_forward", "is_share", "forward_cnt", "share_cnt"],
    "long_view": ["long_view", "is_long_view", "long_play", "longview"],
    "hate": ["hate", "dislike", "is_hate", "is_dislike"],
    "play_time": ["play_time_truncate", "play_time", "play_time_ms", "watch_time", "playing_time"],
    "duration": ["duration", "duration_ms", "video_duration", "item_duration"],
}
OVERRIDES = {}   # e.g. {"rec": {"item": "pos_item_id"}, "src": {"ts": "time_ms"}}

def resolve(cols, role):
    lower = {c.lower(): c for c in cols}
    out = {}
    for canon, cands in CANDIDATES.items():
        if canon in OVERRIDES.get(role, {}):
            out[canon] = OVERRIDES[role][canon]
            continue
        hit = next((lower[c] for c in cands if c in lower), None)
        if hit:
            out[canon] = hit
    return out

colmap = {role: resolve(HEADS[role].columns, role) for role in ("rec", "src") if role in HEADS}
display(pd.DataFrame({r: pd.Series(m) for r, m in colmap.items()}).reindex(CANDIDATES).fillna("—"))
for role, m in colmap.items():
    print(f"{role}: unmapped raw columns ->", [c for c in HEADS[role].columns if c not in m.values()])
    assert {"user", "item", "ts"} <= set(m), f"{role}: need user/item/ts mapped, got {m}"

## 4 · Load interactions

In [ ]:
FLAGS = ["click", "like", "follow", "comment", "forward", "long_view", "hate"]
OBSERVED = {}   # which behaviour flags each source actually logs

def parse_ts(s):
    if pd.api.types.is_numeric_dtype(s):
        med = s.dropna().median()
        unit = "ms" if med > 1e11 else "s" if med > 1e8 else None
        if unit is None:
            raise ValueError(f"Unrecognised numeric timestamp scale (median={med}). Edit parse_ts().")
        return pd.to_datetime(s, unit=unit, errors="coerce")
    return pd.to_datetime(s, errors="coerce")

def load_inter(role, source):
    cm = colmap[role]
    use = list(dict.fromkeys(cm.values()))
    df = pd.concat([read_any(f, usecols=use) for f in FILES[role]], ignore_index=True)
    df = df.rename(columns={v: k for k, v in cm.items()})
    df["ts"] = parse_ts(df["ts"])
    df["source"] = source
    if "query" not in df:
        df["query"] = -1
    OBSERVED[source] = [f for f in FLAGS if f in df]
    print(f"{source:6s}: {len(df):>12,} rows | flags logged: {OBSERVED[source]}")
    return df

parts = []
if FILES["rec"]: parts.append(load_inter("rec", "rec"))
if FILES["src"]: parts.append(load_inter("src", "search"))
events = pd.concat(parts, ignore_index=True, sort=False)
for c in ("user", "item"):
    events[c] = pd.to_numeric(events[c], downcast="integer")
events["source"] = events["source"].astype("category")
print(f"\nTotal events: {len(events):,} | memory: {events.memory_usage(deep=True).sum() / 1e6:,.0f} MB")

## 5 · Load item & user tables

In [ ]:
def load_table(role, key):
    if not FILES.get(role):
        return None
    df = pd.concat([read_any(f) for f in FILES[role]], ignore_index=True)
    lower = {c.lower(): c for c in df.columns}
    key_col = next((lower[c] for c in CANDIDATES[key] if c in lower), None)
    assert key_col, f"{role}: no {key} column among {list(df.columns)[:15]}"
    return df.rename(columns={key_col: key})

items_raw, users_raw = load_table("item", "item"), load_table("user", "user")

CAT_COL = None   # set manually to override, e.g. "first_level_category_name"
items = None
if items_raw is not None:
    cat_cols = [c for c in items_raw.columns if re.search(r"categ|class|tag|genre|level", c, re.I)]
    print("category-like columns:", cat_cols)
    CAT_COL = CAT_COL or next((c for c in cat_cols if re.search(r"first|1|top|primary", c, re.I)),
                              cat_cols[0] if cat_cols else None)
    print("using category column:", CAT_COL)
    items = pd.DataFrame({"item": items_raw["item"]})
    items["cat"] = items_raw[CAT_COL].astype(str).str.strip() if CAT_COL else "unknown"
    items["cat"] = items["cat"].replace({"nan": "unknown", "None": "unknown", "<NA>": "unknown", "": "unknown"})
    lower = {c.lower(): c for c in items_raw.columns}
    dur_col = next((lower[c] for c in CANDIDATES["duration"] if c in lower), None)
    if dur_col:
        items["duration"] = pd.to_numeric(items_raw[dur_col], errors="coerce")
    items = items.drop_duplicates("item")
    print(f"items table: {len(items):,} rows | {items['cat'].nunique()} categories")
else:
    print("No item feature file found: categories will be 'unknown'. Category features need this file.")
print("users table:", None if users_raw is None else users_raw.shape)

## 6 · Profile the raw data

In [ ]:
def profile(df, name, sample=2_000_000):
    s = df.sample(min(len(df), sample), random_state=0) if len(df) > sample else df
    p = pd.DataFrame({"dtype": df.dtypes.astype(str), "nulls": df.isna().sum(),
                      "null_%": (df.isna().mean() * 100).round(2), "nunique~": s.nunique(dropna=True)})
    num = df.select_dtypes("number")
    p = p.join(num.agg(["min", "max"]).T)
    print(f"\n### {name}: {df.shape[0]:,} rows × {df.shape[1]} cols")
    display(p)

profile(events, "events (raw)")
if items is not None: profile(items, "items")
if users_raw is not None: profile(users_raw, "users")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
daily = events.groupby([events["ts"].dt.date, "source"], observed=True).size().unstack(fill_value=0)
daily.plot(kind="bar", stacked=True, ax=ax[0], width=0.85)
ax[0].set(title="Raw events per day (by source)", xlabel="", ylabel="events")
ax[0].tick_params(axis="x", rotation=60)

prev = {f: [events.loc[events["source"] == s, f].gt(0).mean() * 100 if f in OBSERVED[s] else np.nan
            for s in OBSERVED] for f in FLAGS if any(f in v for v in OBSERVED.values())}
pd.DataFrame(prev, index=list(OBSERVED)).T.plot(kind="barh", ax=ax[1])
ax[1].set(title="Behaviour prevalence (% of events; blank = not logged by that source)", xlabel="%")
savefig("01_raw_volume_and_prevalence")

## 7 · Data-quality audit (before any change)

In [ ]:
flags_present = [f for f in FLAGS if f in events]
strong = [f for f in ("like", "comment", "forward", "follow", "long_view") if f in events]
rows = [("total rows", len(events)),
        ("exact duplicate rows", int(events.duplicated().sum())),
        ("null user", int(events["user"].isna().sum())),
        ("null item", int(events["item"].isna().sum())),
        ("null/unparseable timestamp", int(events["ts"].isna().sum())),
        ("timestamp before 2015 or in the future", int(((events["ts"] < "2015-01-01") | (events["ts"] > pd.Timestamp.now())).sum()))]
for f in flags_present:
    v = events[f].dropna()
    rows.append((f"flag '{f}' not in {{0,1}} (will be binarised)", int((~v.isin([0, 1])).sum())))
if "click" in events and strong:
    rows.append(("strong action without click (impossible?)", int(((events[strong].fillna(0).sum(axis=1) > 0) & (events["click"].fillna(0) == 0)).sum())))
if "play_time" in events: rows.append(("negative play_time", int((events["play_time"] < 0).sum())))
if "duration" in events: rows.append(("duration <= 0", int((events["duration"] <= 0).sum())))
if users_raw is not None: rows.append(("events from users missing in user table", int((~events["user"].isin(users_raw["user"])).sum())))
if items is not None:
    rows.append(("events on items missing in item table", int((~events["item"].isin(items["item"])).sum())))
    rows.append(("items with unknown category", int((items["cat"] == "unknown").sum())))
audit = pd.DataFrame(rows, columns=["check", "count"])
audit["% of events"] = (audit["count"] / len(events) * 100).round(3)
audit

## 8 · Clean
Each step logs `rows_before / rows_after`. Behaviour toggles are at the top so the team can agree on them once.

* `FIX_IMPLIED_CLICK`: if a like/share/follow exists but click = 0, set click = 1. Off by default because in feed products a like can happen without a detail-page "click".
* `DROP_EXTREME_USERS`: remove users above the given activity quantile (likely bots). Off by default; review the table first.

In [ ]:
CFG = dict(FIX_IMPLIED_CLICK=False, DROP_EXTREME_USERS=False, EXTREME_USER_Q=0.9999, MIN_VALID_DATE="2015-01-01")
log = []
def step(name, before, after, note=""):
    log.append(dict(step=name, rows_before=before, rows_after=after, removed=before - after, note=note))

ev = events
n = len(ev)

ev = ev.dropna(subset=["user", "item", "ts"]); step("drop null user/item/ts", n, len(ev)); n = len(ev)

ev = ev[(ev["ts"] >= CFG["MIN_VALID_DATE"]) & (ev["ts"] <= pd.Timestamp.now())]
step("drop impossible timestamps", n, len(ev)); n = len(ev)

for f in flags_present:                                    # counts / NaN -> clean 0/1
    ev[f] = (ev[f].fillna(0) > 0).astype("int8")
step("binarise behaviour flags", n, len(ev), "NaN->0, counts->1; flags a source does not log stay 0 (see schema.json)")

ev = ev.drop_duplicates(); step("drop exact duplicate rows", n, len(ev)); n = len(ev)

if CFG["FIX_IMPLIED_CLICK"] and "click" in ev and strong:
    m = (ev[strong].sum(axis=1) > 0) & (ev["click"] == 0)
    ev.loc[m, "click"] = 1; step("set implied click", n, len(ev), f"{int(m.sum()):,} rows modified")

if "duration" not in ev and items is not None and "duration" in items:
    ev["duration"] = ev["item"].map(items.set_index("item")["duration"])
if "play_time" in ev:
    ev.loc[ev["play_time"] < 0, "play_time"] = np.nan
if "duration" in ev:
    ev.loc[ev["duration"] <= 0, "duration"] = np.nan
if {"play_time", "duration"} <= set(ev.columns):
    ratio = (ev["play_time"] / ev["duration"]).replace([np.inf, -np.inf], np.nan)
    med = ratio.median()
    print(f"median play_time/duration = {med:.3f}", "<- check units!" if not (0.01 < med < 5) else "")
    ev["play_ratio"] = ratio.clip(0, 5).astype("float32")
    step("sanitise play_time/duration -> play_ratio (clip 0..5)", n, len(ev), "negative/invalid -> NaN")

uc = ev["user"].value_counts()
thr = uc.quantile(CFG["EXTREME_USER_Q"])
extreme = uc[uc > thr]
print(f"users above q={CFG['EXTREME_USER_Q']}: {len(extreme)} (>{thr:.0f} events) | single-event users: {(uc == 1).sum():,}")
if CFG["DROP_EXTREME_USERS"]:
    ev = ev[~ev["user"].isin(extreme.index)]; step("drop extreme users", n, len(ev)); n = len(ev)

ev = ev.sort_values("ts").reset_index(drop=True)
for c in ("user", "item"):
    ev[c] = pd.to_numeric(ev[c], downcast="integer")

report = pd.DataFrame(log); report.to_csv(INTERIM / "cleaning_report.csv", index=False)
display(report)
print(f"\nFinal: {len(ev):,} rows ({len(ev) / len(events) * 100:.2f}% of raw) | users {ev['user'].nunique():,} | items {ev['item'].nunique():,}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
r = report[report["removed"] > 0]
ax[0].barh(r["step"], r["removed"]) if len(r) else ax[0].text(0.5, 0.5, "no rows removed", ha="center")
ax[0].set(title="Rows removed per cleaning step", xlabel="rows"); ax[0].invert_yaxis()
uc.clip(upper=uc.quantile(0.999)).plot(kind="hist", bins=60, ax=ax[1], log=True)
ax[1].set(title="Events per user (clipped at p99.9, log y)", xlabel="events")
savefig("01_cleaning_summary")

## 9 · Save

In [ ]:
def to_parquet_safe(df, path):
    df = df.copy()
    for c in df.select_dtypes(["object", "string"]).columns:
        df[c] = df[c].astype(str)
    df.to_parquet(path, index=False)

ev["source"] = ev["source"].astype("category")
to_parquet_safe(ev, INTERIM / "events.parquet")
if items is not None: to_parquet_safe(items, INTERIM / "items.parquet")
if users_raw is not None: to_parquet_safe(users_raw, INTERIM / "users.parquet")

schema = dict(flags_present=[f for f in FLAGS if f in ev], observed_by_source=OBSERVED, cat_col=CAT_COL,
              colmap=colmap, n_events=int(len(ev)), n_users=int(ev["user"].nunique()), n_items=int(ev["item"].nunique()),
              ts_min=str(ev["ts"].min()), ts_max=str(ev["ts"].max()), cleaning=CFG)
json.dump(schema, open(INTERIM / "schema.json", "w"), indent=2, default=str)
print(json.dumps({k: schema[k] for k in ("flags_present", "observed_by_source", "n_events", "ts_min", "ts_max")}, indent=2))
print("\nSaved to", INTERIM)

## Hand-off to notebook 02
* `events.parquet` — one row per logged interaction with canonical columns, binary behaviour flags, `source` (rec/search) and optional `play_ratio`.
* Behaviours a source does **not** log are 0, not "unknown": check `schema.json → observed_by_source` before drawing conclusions about them.
* Decisions the team should note in the report: `CFG` toggles, category column used, and any rows dropped (see `cleaning_report.csv`).